In [ ]:
import requests
import pandas as pd
import json
import time
from typing import Any
import pymysql
from urllib.parse import quote_plus
from sqlalchemy import create_engine

# MySQL connection details (update if necessary)
DB_USER = "root"
DB_PASSWORD = quote_plus("Vasu@76652")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "cricketdb"

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4")

# Load team IDs from the teams table
try:
    query = "SELECT Team_ID FROM teams ORDER BY Team_ID LIMIT 10000;"
    df_teams = pd.read_sql(query, engine)
    team_id = df_teams['Team_ID'].tolist() if not df_teams.empty else []
    print(f"Found {len(team_id)} teams")
except Exception as e:
    print(f"Failed to fetch team IDs from DB: {e}")
    team_id = []

all_data = []
DEBUG_MODE = True  # Set to False to disable detailed logging
for TEAM_ID in team_id[:1]:  # Limit to first team for debugging
    RAPID_API_KEY = "01dee2433cmsh3c82e01c3bf483fp18b648jsnd30d7386c46a"

    url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{TEAM_ID}/players"

    headers = {
        "x-rapidapi-key": RAPID_API_KEY,
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to fetch team {TEAM_ID}: {response.status_code}")
        continue

    data = response.json()
    players = []

    if "player" in data:
        for idx, p in enumerate(data["player"][:3]):  # Limit to first 3 players for debugging
            player_id = p.get("id")
            
            if DEBUG_MODE and idx == 0:
                print(f"\n--- DEBUG: First player from team endpoint ---")
                print(f"Player ID: {player_id}")
                print(f"Available keys in team endpoint: {p.keys()}")
                print(f"Full data: {json.dumps(p, indent=2)}\n")
            
            # Fetch detailed player info using player stats API
            if player_id:
                try:
                    player_detail_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"
                    player_detail_response = requests.get(player_detail_url, headers=headers)
                    
                    if player_detail_response.status_code == 200:
                        player_detail = player_detail_response.json()
                        
                        if DEBUG_MODE and idx == 0:
                            print(f"--- DEBUG: Detailed player endpoint response for ID {player_id} ---")
                            print(f"Available keys: {player_detail.keys()}")
                            print(f"Full response: {json.dumps(player_detail, indent=2)}\n")
                        
                        # Try to extract fields - check multiple possible locations
                        player_info = {
                            "player_id": player_id,
                            "player_name": player_detail.get("name") or p.get("name"),
                            "country": player_detail.get("country") or player_detail.get("countryName"),
                            "role": player_detail.get("role") or p.get("role"),
                            "batting_style": player_detail.get("battingStyle") or player_detail.get("battingHand"),
                            "bowling_style": player_detail.get("bowlingStyle"),
                            "dob": player_detail.get("dateOfBirth") or player_detail.get("DOB")
                        }
                        players.append(player_info)
                    else:
                        print(f"Failed to fetch detailed info for player {player_id}: {player_detail_response.status_code}")
                        players.append({
                            "player_id": player_id,
                            "player_name": p.get("name"),
                            "country": p.get("country"),
                            "role": p.get("role"),
                            "batting_style": p.get("battingStyle"),
                            "bowling_style": p.get("bowlingStyle"),
                            "dob": p.get("dateOfBirth")
                        })
                except Exception as detail_err:
                    print(f"Error fetching detailed info for player {player_id}: {detail_err}")
                    players.append({
                        "player_id": player_id,
                        "player_name": p.get("name"),
                        "country": p.get("country"),
                        "role": p.get("role"),
                        "batting_style": p.get("battingStyle"),
                        "bowling_style": p.get("bowlingStyle"),
                        "dob": p.get("dateOfBirth")
                    })
            else:
                players.append({
                    "player_id": player_id,
                    "player_name": p.get("name"),
                    "country": p.get("country"),
                    "role": p.get("role"),
                    "batting_style": p.get("battingStyle"),
                    "bowling_style": p.get("bowlingStyle"),
                    "dob": p.get("dateOfBirth")
                })
            
            # Delay between API calls to avoid rate limiting
            time.sleep(0.1)

    print(f"Total Players Fetched: {len(players)}")
    print(f"Sample player data:")
    if players:
        print(json.dumps(players[0], indent=2))

    sql = "INSERT INTO players (Player_ID, Player_Name, Country, Role, Batting_Style, Bowling_Style, DOB, Meta) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON DUPLICATE KEY UPDATE Player_Name=VALUES(Player_Name), Country=VALUES(Country), Role=VALUES(Role), Batting_Style=VALUES(Batting_Style), Bowling_Style=VALUES(Bowling_Style), DOB=VALUES(DOB), Meta=VALUES(Meta)"

    # Use engine's raw DBAPI connection for executing parameterized inserts
    if hasattr(engine, "raw_connection"):
        conn = engine.raw_connection()  # type: ignore
    else:
        conn = engine

    cursor = conn.cursor()
    try:
        for p in players:
            meta = json.dumps(p)
            params = (p.get("player_id"), p.get("player_name"), p.get("country"), p.get("role"), p.get("batting_style"), p.get("bowling_style"), p.get("dob"), meta)
            cursor.execute(sql, params)
        conn.commit()
        print("Players inserted/updated successfully!")
    finally:
        try:
            cursor.close()
        except Exception:
            pass
        try:
            conn.close()
        except Exception:
            pass

    all_data.append({"team_id": TEAM_ID, "players": data})
    print(TEAM_ID, data.keys())


Found 924 teams
Total Players Fetched: 35
Players inserted/updated successfully!
10 dict_keys(['player'])
Total Players Fetched: 19
Players inserted/updated successfully!
100 dict_keys(['player'])
Failed to fetch team 101: 204
Failed to fetch team 1011: 204
Failed to fetch team 1018: 204
Failed to fetch team 1020: 204
Failed to fetch team 1027: 204
Failed to fetch team 103: 204
Failed to fetch team 1032: 204
Failed to fetch team 1034: 204
Failed to fetch team 104: 204
Failed to fetch team 1041: 204
Failed to fetch team 1048: 204
Failed to fetch team 105: 204
Failed to fetch team 1055: 204
Failed to fetch team 1062: 204
Failed to fetch team 1067: 204
Failed to fetch team 1069: 204
Failed to fetch team 1076: 204
Failed to fetch team 1081: 204
Failed to fetch team 1083: 204
Failed to fetch team 1090: 204
Failed to fetch team 1097: 204
Total Players Fetched: 32
Players inserted/updated successfully!
11 dict_keys(['player'])
Failed to fetch team 1104: 204
Failed to fetch team 1111: 204
Fail

KeyboardInterrupt: 

In [ ]:
c

Found 924 teams
Found 192 existing players in DB

Processing Team: 10
Skipping existing player 9791
Skipping existing player 13352
Skipping existing player 1984
Skipping existing player 8434
Skipping existing player 11445
Skipping existing player 9789
Skipping existing player 8579
Skipping existing player 11217
Skipping existing player 8433
Skipping existing player 13748
Skipping existing player 8431
Skipping existing player 9793
Skipping existing player 11543
Skipping existing player 13646
Skipping existing player 8313
Skipping existing player 7736
Skipping existing player 10384
Skipping existing player 16414
Skipping existing player 8101
Skipping existing player 1394
Skipping existing player 54287
Skipping existing player 12674
Skipping existing player 11220
Skipping existing player 50458
Skipping existing player 15817
Skipping existing player 8435
Skipping existing player 13958
Skipping existing player 11221
Skipping existing player 11223
Skipping existing player 15809
Skipping exis

In [6]:
import requests
import pandas as pd
import json
import time
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime

# ==============================
# CONFIG
# ==============================

RAPID_API_KEY = "0b0e099125mshb85e67ba0b73081p157c9bjsn62ebc6d869d1"

DB_USER = "root"
DB_PASSWORD = quote_plus("Vasu@76652")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "cricketdb"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True
)

session = requests.Session()

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# ==============================
# HELPER FUNCTIONS
# ==============================

def clean_dob(dob_string):
    if not dob_string:
        return None
    try:
        date_part = dob_string.split("(")[0].strip()
        parsed_date = datetime.strptime(date_part, "%B %d, %Y")
        return parsed_date.strftime("%Y-%m-%d")
    except:
        return None


def safe_api_call(url, retries=5):
    """
    Handles 429 rate limit with exponential backoff
    """
    for attempt in range(retries):
        response = session.get(url, headers=headers, timeout=20)

        if response.status_code == 429:
            wait_time = 2 ** attempt
            print(f"Rate limited. Waiting {wait_time} sec...")
            time.sleep(wait_time)
            continue

        try:
            response.raise_for_status()
            return response.json()
        except:
            print(f"API error: {response.status_code}")
            return None

    print("Max retries reached. Skipping...")
    return None


# ==============================
# LOAD DATA FROM DB
# ==============================

df_teams = pd.read_sql("SELECT Team_ID FROM teams ORDER BY Team_ID", engine)
team_ids = df_teams["Team_ID"].astype(str).tolist()
print(f"Total teams: {len(team_ids)}")

df_existing = pd.read_sql("SELECT Player_ID FROM players", engine)
existing_players = set(df_existing["Player_ID"].astype(str))
print(f"Existing players: {len(existing_players)}")

# ==============================
# MAIN LOOP
# ==============================

for TEAM_ID in team_ids:

    print(f"\nProcessing Team: {TEAM_ID}")

    team_url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{TEAM_ID}/players"
    team_data = safe_api_call(team_url)

    if not team_data or "player" not in team_data:
        continue

    total_players_api = len(team_data["player"])

    # 🔥 Check how many players already mapped for this team
    with engine.connect() as conn:
        result = conn.exec_driver_sql(
            "SELECT COUNT(*) FROM player_team WHERE Team_ID = %s",
            (TEAM_ID,)
        )
        existing_count = result.scalar()

    if existing_count >= total_players_api:
        print(f"Team {TEAM_ID} already fully processed. Skipping.")
        continue

    print(f"API players: {total_players_api}, DB mapped: {existing_count}")

    players_batch = []

    for p in team_data["player"]:

        player_id_raw = p.get("id")

        if not player_id_raw or not str(player_id_raw).isdigit():
            continue

        player_id = str(player_id_raw)

        # If already mapped for this team, skip
        with engine.connect() as conn:
            result = conn.exec_driver_sql(
                "SELECT 1 FROM player_team WHERE Player_ID=%s AND Team_ID=%s LIMIT 1",
                (player_id, TEAM_ID)
            )
            if result.fetchone():
                continue

        # If player exists globally, only mapping needed
        if player_id in existing_players:
            players_batch.append({
                "player_id": player_id,
                "existing_only": True
            })
            continue

        # Fetch detailed player info
        detail_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"
        detail = safe_api_call(detail_url)

        if not detail:
            continue

        player_info = {
            "player_id": player_id,
            "player_name": detail.get("name") or p.get("name"),
            "country": detail.get("country") or detail.get("intlTeam"),
            "role": detail.get("role") or p.get("role"),
            "batting_style": detail.get("battingStyle") or p.get("battingStyle"),
            "bowling_style": detail.get("bowlingStyle") or p.get("bowlingStyle"),
            "dob": clean_dob(detail.get("DoB"))
        }

        players_batch.append(player_info)
        existing_players.add(player_id)

        time.sleep(0.2)

    # ==============================
    # DATABASE INSERT
    # ==============================

    insert_player_sql = """
    INSERT INTO players 
    (Player_ID, Player_Name, Country, Role, Batting_Style, Bowling_Style, DOB, Meta)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        Player_Name=VALUES(Player_Name),
        Country=VALUES(Country),
        Role=VALUES(Role),
        Batting_Style=VALUES(Batting_Style),
        Bowling_Style=VALUES(Bowling_Style),
        DOB=VALUES(DOB),
        Meta=VALUES(Meta)
    """

    insert_mapping_sql = """
    INSERT IGNORE INTO player_team (Player_ID, Team_ID)
    VALUES (%s, %s)
    """

    with engine.begin() as conn:

        for player in players_batch:

            player_id = player["player_id"]

            if not player.get("existing_only"):

                conn.exec_driver_sql(insert_player_sql, (
                    player["player_id"],
                    player.get("player_name"),
                    player.get("country"),
                    player.get("role"),
                    player.get("batting_style"),
                    player.get("bowling_style"),
                    player.get("dob"),
                    json.dumps(player)
                ))

            conn.exec_driver_sql(insert_mapping_sql, (
                player_id,
                TEAM_ID
            ))

    print(f"Team {TEAM_ID} processed successfully")

print("\nAll teams processed successfully 🚀")


Total teams: 924
Existing players: 245

Processing Team: 10
API players: 35, DB mapped: 0
Team 10 processed successfully

Processing Team: 100
API players: 19, DB mapped: 0
Team 100 processed successfully

Processing Team: 101
API error: 204

Processing Team: 1011
API error: 204

Processing Team: 1018
API error: 204

Processing Team: 1020
API error: 204

Processing Team: 1027
API error: 204

Processing Team: 103
API error: 204

Processing Team: 1032
API error: 204

Processing Team: 1034
API error: 204

Processing Team: 104
API error: 204

Processing Team: 1041
API error: 204

Processing Team: 1048
API error: 204

Processing Team: 105
API error: 204

Processing Team: 1055
API error: 204

Processing Team: 1062
API error: 204

Processing Team: 1067
API error: 204

Processing Team: 1069
API error: 204

Processing Team: 1076
API error: 204

Processing Team: 1081
API error: 204

Processing Team: 1083
API error: 204

Processing Team: 1090
API error: 204

Processing Team: 1097
API error: 204



KeyboardInterrupt: 

In [10]:
import requests
import pandas as pd
import json
import time
import sys
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime

# ==============================
# CONFIGURATION
# ==============================

RAPID_API_KEY = "08014320a4mshad5df2e0e8b7ab0p11a251jsn838428db6716"

DB_USER = "root"
DB_PASSWORD = quote_plus("Vasu@76652")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "cricketdb"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True
)

session = requests.Session()

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# ==============================
# HELPER FUNCTIONS
# ==============================

def clean_dob(dob_string):
    if not dob_string:
        return None
    try:
        date_part = dob_string.split("(")[0].strip()
        parsed_date = datetime.strptime(date_part, "%B %d, %Y")
        return parsed_date.strftime("%Y-%m-%d")
    except:
        return None


def safe_api_call(url):
    response = session.get(url, headers=headers, timeout=20)

    if response.status_code == 429:
        print("\n🚨 RATE LIMIT (429) HIT. Stopping program safely.")
        return "RATE_LIMIT"

    if response.status_code == 204:
        # No content – team has no squad
        return {}

    try:
        response.raise_for_status()
        return response.json()
    except:
        print(f"API error: {response.status_code}")
        return None


# ==============================
# LOAD DATA FROM DB
# ==============================

df_teams = pd.read_sql("SELECT Team_ID FROM teams ORDER BY Team_ID", engine)
team_ids = df_teams["Team_ID"].astype(str).tolist()
print(f"Total teams: {len(team_ids)}")

df_existing = pd.read_sql("SELECT Player_ID FROM players", engine)
existing_players = set(df_existing["Player_ID"].astype(str))
print(f"Existing players: {len(existing_players)}")

# ==============================
# MAIN LOOP
# ==============================

for TEAM_ID in team_ids:

    print(f"\nProcessing Team: {TEAM_ID}")

    team_url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{TEAM_ID}/players"
    team_data = safe_api_call(team_url)

    if team_data == "RATE_LIMIT":
        print("Program stopped due to rate limit.")
        sys.exit(0)

    if not team_data or "player" not in team_data:
        continue

    total_players_api = len(team_data["player"])

    # Check how many players already mapped for this team
    with engine.connect() as conn:
        result = conn.exec_driver_sql(
            "SELECT COUNT(*) FROM player_team WHERE Team_ID = %s",
            (TEAM_ID,)
        )
        existing_count = result.scalar()

    # Skip if fully processed
    if existing_count >= total_players_api:
        print(f"Team {TEAM_ID} already fully processed. Skipping.")
        continue

    players_batch = []

    for p in team_data["player"]:

        player_id_raw = p.get("id")

        if not player_id_raw or not str(player_id_raw).isdigit():
            continue

        player_id = str(player_id_raw)

        # Skip if mapping already exists
        with engine.connect() as conn:
            result = conn.exec_driver_sql(
                "SELECT 1 FROM player_team WHERE Player_ID=%s AND Team_ID=%s LIMIT 1",
                (player_id, TEAM_ID)
            )
            if result.fetchone():
                continue

        # If player exists globally, only mapping needed
        if player_id in existing_players:
            players_batch.append({
                "player_id": player_id,
                "existing_only": True
            })
            continue

        # Fetch detailed info
        detail_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"
        detail = safe_api_call(detail_url)

        if detail == "RATE_LIMIT":
            print("Stopping safely due to rate limit.")
            break

        if not detail:
            continue

        player_info = {
            "player_id": player_id,
            "player_name": detail.get("name") or p.get("name"),
            "country": detail.get("country") or detail.get("intlTeam"),
            "role": detail.get("role") or p.get("role"),
            "batting_style": detail.get("battingStyle") or p.get("battingStyle"),
            "bowling_style": detail.get("bowlingStyle") or p.get("bowlingStyle"),
            "dob": clean_dob(detail.get("DoB"))
        }

        players_batch.append(player_info)
        existing_players.add(player_id)

        time.sleep(0.2)

    # ==============================
    # INSERT COLLECTED DATA
    # ==============================

    insert_player_sql = """
    INSERT INTO players 
    (Player_ID, Player_Name, Country, Role, Batting_Style, Bowling_Style, DOB, Meta)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        Player_Name=VALUES(Player_Name),
        Country=VALUES(Country),
        Role=VALUES(Role),
        Batting_Style=VALUES(Batting_Style),
        Bowling_Style=VALUES(Bowling_Style),
        DOB=VALUES(DOB),
        Meta=VALUES(Meta)
    """

    insert_mapping_sql = """
    INSERT IGNORE INTO player_team (Player_ID, Team_ID)
    VALUES (%s, %s)
    """

    if players_batch:

        with engine.begin() as conn:
            for player in players_batch:

                player_id = player["player_id"]

                if not player.get("existing_only"):
                    conn.exec_driver_sql(insert_player_sql, (
                        player["player_id"],
                        player.get("player_name"),
                        player.get("country"),
                        player.get("role"),
                        player.get("batting_style"),
                        player.get("bowling_style"),
                        player.get("dob"),
                        json.dumps(player)
                    ))

                conn.exec_driver_sql(insert_mapping_sql, (
                    player_id,
                    TEAM_ID
                ))

        print(f"Inserted data for Team {TEAM_ID}")

print("\nAll teams processed safely 🚀")


Total teams: 924
Existing players: 262

Processing Team: 10

Processing Team: 100

Processing Team: 101

Processing Team: 1011

Processing Team: 1018

Processing Team: 1020

Processing Team: 1027

Processing Team: 103

Processing Team: 1032

Processing Team: 1034

Processing Team: 104

Processing Team: 1041

Processing Team: 1048

Processing Team: 105

Processing Team: 1055

Processing Team: 1062

Processing Team: 1067

Processing Team: 1069

Processing Team: 1076

Processing Team: 1081

Processing Team: 1083

Processing Team: 1090

Processing Team: 1097

Processing Team: 11

Processing Team: 1104

Processing Team: 1111

Processing Team: 1116

Processing Team: 1123

Processing Team: 1125

Processing Team: 1132

Processing Team: 1137

Processing Team: 1139

Processing Team: 1144

Processing Team: 1146

Processing Team: 1151

Processing Team: 1153

Processing Team: 1158

Processing Team: 1165

Processing Team: 1172

Processing Team: 1179

Processing Team: 1181

Processing Team: 1188

Pro

SystemExit: 0

C:\Users\Abishek\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [11]:
import requests
import pandas as pd
import json
import time
import sys
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime

# ==============================
# CONFIGURATION
# ==============================

RAPID_API_KEY = "0be62878demshcfd4a74416cba00p1abe24jsn71b03a98d5de"

DB_USER = "root"
DB_PASSWORD = quote_plus("Vasu@76652")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "cricketdb"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True
)

session = requests.Session()

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# ==============================
# HELPER FUNCTIONS
# ==============================

def clean_dob(dob_string):
    if not dob_string:
        return None
    try:
        date_part = dob_string.split("(")[0].strip()
        parsed_date = datetime.strptime(date_part, "%B %d, %Y")
        return parsed_date.strftime("%Y-%m-%d")
    except:
        return None


def safe_api_call(url):
    response = session.get(url, headers=headers, timeout=20)

    if response.status_code == 429:
        return "RATE_LIMIT"

    if response.status_code == 204:
        return {}

    try:
        response.raise_for_status()
        return response.json()
    except:
        return None


# ==============================
# LOAD DATABASE DATA
# ==============================

df_teams = pd.read_sql("SELECT Team_ID FROM teams ORDER BY Team_ID", engine)
team_ids = df_teams["Team_ID"].astype(str).tolist()

df_existing = pd.read_sql("SELECT Player_ID FROM players", engine)
existing_players = set(df_existing["Player_ID"].astype(str))

print(f"Total teams: {len(team_ids)}")
print(f"Existing players: {len(existing_players)}")

# ==============================
# SQL QUERIES
# ==============================

insert_player_sql = """
INSERT INTO players 
(Player_ID, Player_Name, Country, Role, Batting_Style, Bowling_Style, DOB, Meta)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    Player_Name=VALUES(Player_Name),
    Country=VALUES(Country),
    Role=VALUES(Role),
    Batting_Style=VALUES(Batting_Style),
    Bowling_Style=VALUES(Bowling_Style),
    DOB=VALUES(DOB),
    Meta=VALUES(Meta)
"""

insert_mapping_sql = """
INSERT IGNORE INTO player_team (Player_ID, Team_ID)
VALUES (%s, %s)
"""

# ==============================
# MAIN PROCESS
# ==============================

rate_limit_hit = False

for TEAM_ID in team_ids:

    print(f"\nProcessing Team: {TEAM_ID}")

    team_url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{TEAM_ID}/players"
    team_data = safe_api_call(team_url)

    if team_data == "RATE_LIMIT":
        print("🚨 429 hit while fetching team list")
        rate_limit_hit = True
        break

    if not team_data or "player" not in team_data:
        continue

    players_batch = []

    for p in team_data["player"]:

        player_id_raw = p.get("id")

        if not player_id_raw or not str(player_id_raw).isdigit():
            continue

        player_id = str(player_id_raw)

        # Skip if mapping already exists
        with engine.connect() as conn:
            result = conn.exec_driver_sql(
                "SELECT 1 FROM player_team WHERE Player_ID=%s AND Team_ID=%s LIMIT 1",
                (player_id, TEAM_ID)
            )
            if result.fetchone():
                continue

        # Existing player → only mapping needed
        if player_id in existing_players:
            players_batch.append({
                "player_id": player_id,
                "existing_only": True
            })
            continue

        # Fetch detail
        detail_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"
        detail = safe_api_call(detail_url)

        if detail == "RATE_LIMIT":
            print("🚨 429 hit while fetching player detail")
            rate_limit_hit = True
            break

        if not detail:
            continue

        player_info = {
            "player_id": player_id,
            "player_name": detail.get("name") or p.get("name"),
            "country": detail.get("country") or detail.get("intlTeam"),
            "role": detail.get("role") or p.get("role"),
            "batting_style": detail.get("battingStyle") or p.get("battingStyle"),
            "bowling_style": detail.get("bowlingStyle") or p.get("bowlingStyle"),
            "dob": clean_dob(detail.get("DoB"))
        }

        players_batch.append(player_info)
        existing_players.add(player_id)

        time.sleep(0.2)

    # 🔥 INSERT COLLECTED PLAYERS (ALWAYS INSERT BEFORE STOP)
    if players_batch:

        with engine.begin() as conn:
            for player in players_batch:

                player_id = player["player_id"]

                if not player.get("existing_only"):
                    conn.exec_driver_sql(insert_player_sql, (
                        player["player_id"],
                        player.get("player_name"),
                        player.get("country"),
                        player.get("role"),
                        player.get("batting_style"),
                        player.get("bowling_style"),
                        player.get("dob"),
                        json.dumps(player)
                    ))

                conn.exec_driver_sql(insert_mapping_sql, (
                    player_id,
                    TEAM_ID
                ))

        print(f"Inserted collected data for Team {TEAM_ID}")

    # If 429 occurred, stop AFTER inserting
    if rate_limit_hit:
        print("Program stopped safely after inserting collected data.")
        sys.exit(0)

print("\nAll teams processed successfully 🚀")


Total teams: 924
Existing players: 262

Processing Team: 10

Processing Team: 100

Processing Team: 101

Processing Team: 1011

Processing Team: 1018

Processing Team: 1020

Processing Team: 1027

Processing Team: 103

Processing Team: 1032

Processing Team: 1034

Processing Team: 104

Processing Team: 1041

Processing Team: 1048

Processing Team: 105

Processing Team: 1055

Processing Team: 1062

Processing Team: 1067

Processing Team: 1069

Processing Team: 1076

Processing Team: 1081

Processing Team: 1083

Processing Team: 1090

Processing Team: 1097

Processing Team: 11

Processing Team: 1104

Processing Team: 1111

Processing Team: 1116

Processing Team: 1123

Processing Team: 1125

Processing Team: 1132

Processing Team: 1137

Processing Team: 1139

Processing Team: 1144

Processing Team: 1146

Processing Team: 1151

Processing Team: 1153

Processing Team: 1158

Processing Team: 1165

Processing Team: 1172

Processing Team: 1179

Processing Team: 1181

Processing Team: 1188

Pro

In [2]:
import requests
import pandas as pd
import json
import time
import sys
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime

# ==============================
# CONFIGURATION
# ==============================

RAPID_API_KEY = "8cbcf80c10mshbf743949673a7f7p14bd04jsndcb1f2bf1b6c"

DB_USER = "root"
DB_PASSWORD = quote_plus("Vasu@76652")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "cricketdb"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True
)

session = requests.Session()

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# ==============================
# HELPER FUNCTIONS
# ==============================

def clean_dob(dob_string):
    if not dob_string:
        return None
    try:
        date_part = dob_string.split("(")[0].strip()
        parsed_date = datetime.strptime(date_part, "%B %d, %Y")
        return parsed_date.strftime("%Y-%m-%d")
    except:
        return None


def safe_api_call(url):
    try:
        response = session.get(url, headers=headers, timeout=20)

        if response.status_code == 429:
            return "RATE_LIMIT"

        if response.status_code == 204:
            return {}

        response.raise_for_status()
        return response.json()

    except:
        return None


def team_fully_collected(team_id, total_players):
    with engine.connect() as conn:
        result = conn.exec_driver_sql(
            "SELECT COUNT(*) FROM player_team WHERE Team_ID=%s",
            (team_id,)
        )
        collected = result.scalar()
        return collected >= total_players


# ==============================
# LOAD DATA
# ==============================

df_teams = pd.read_sql("SELECT Team_ID FROM teams ORDER BY Team_ID", engine)
team_ids = df_teams["Team_ID"].astype(str).tolist()

df_existing = pd.read_sql("SELECT Player_ID FROM players", engine)
existing_players = set(df_existing["Player_ID"].astype(str))

print(f"Total Teams: {len(team_ids)}")
print(f"Existing Players: {len(existing_players)}")

# ==============================
# SQL
# ==============================

insert_player_sql = """
INSERT INTO players
(Player_ID, Player_Name, Country, Role, Batting_Style, Bowling_Style, DOB, Meta)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    Player_Name=VALUES(Player_Name),
    Country=VALUES(Country),
    Role=VALUES(Role),
    Batting_Style=VALUES(Batting_Style),
    Bowling_Style=VALUES(Bowling_Style),
    DOB=VALUES(DOB),
    Meta=VALUES(Meta)
"""

insert_mapping_sql = """
INSERT IGNORE INTO player_team (Player_ID, Team_ID)
VALUES (%s, %s)
"""

# ==============================
# MAIN LOOP
# ==============================

for TEAM_ID in team_ids:

    print(f"\nProcessing Team: {TEAM_ID}")

    team_url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{TEAM_ID}/players"
    team_data = safe_api_call(team_url)

    # 429 at team level
    if team_data == "RATE_LIMIT":
        print("🚨 429 Hit. Stopping safely.")
        sys.exit(0)

    # Skip 204 / empty
    if not team_data or "player" not in team_data or not team_data["player"]:
        print("Skipped (No players)")
        continue

    total_players = len(team_data["player"])

    # Skip fully collected team
    if team_fully_collected(TEAM_ID, total_players):
        print("Skipped (Already fully collected)")
        continue

    players_batch = []

    for p in team_data["player"]:

        player_id_raw = p.get("id")
        if not player_id_raw or not str(player_id_raw).isdigit():
            continue

        player_id = str(player_id_raw)

        # Skip if mapping already exists
        with engine.connect() as conn:
            result = conn.exec_driver_sql(
                "SELECT 1 FROM player_team WHERE Player_ID=%s AND Team_ID=%s LIMIT 1",
                (player_id, TEAM_ID)
            )
            if result.fetchone():
                continue

        # Fetch detail only if player not exists
        if player_id not in existing_players:

            detail_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"
            detail = safe_api_call(detail_url)

            if detail == "RATE_LIMIT":
                print("🚨 429 Hit during player detail. Inserting collected players.")
                break

            if not detail:
                continue

            player_info = {
                "player_id": player_id,
                "player_name": detail.get("name") or p.get("name"),
                "country": detail.get("country") or detail.get("intlTeam"),
                "role": detail.get("role") or p.get("role"),
                "batting_style": detail.get("battingStyle") or p.get("battingStyle"),
                "bowling_style": detail.get("bowlingStyle") or p.get("bowlingStyle"),
                "dob": clean_dob(detail.get("DoB"))
            }

            players_batch.append(player_info)
            existing_players.add(player_id)

        else:
            players_batch.append({
                "player_id": player_id,
                "existing_only": True
            })

        time.sleep(0.2)

    # ==========================
    # INSERT COLLECTED DATA
    # ==========================

    if players_batch:

        with engine.begin() as conn:
            for player in players_batch:

                pid = player["player_id"]

                if not player.get("existing_only"):
                    conn.exec_driver_sql(insert_player_sql, (
                        player["player_id"],
                        player.get("player_name"),
                        player.get("country"),
                        player.get("role"),
                        player.get("batting_style"),
                        player.get("bowling_style"),
                        player.get("dob"),
                        json.dumps(player)
                    ))

                conn.exec_driver_sql(insert_mapping_sql, (pid, TEAM_ID))

        print(f"Inserted {len(players_batch)} players/mappings")

print("\nAll teams processed successfully 🚀")


Total Teams: 924
Existing Players: 262

Processing Team: 10

Processing Team: 100

Processing Team: 101
Skipped (No players)

Processing Team: 1011
Skipped (No players)

Processing Team: 1018
Skipped (No players)

Processing Team: 1020
Skipped (No players)

Processing Team: 1027
Skipped (No players)

Processing Team: 103
Skipped (No players)

Processing Team: 1032
Skipped (No players)

Processing Team: 1034
Skipped (No players)

Processing Team: 104
Skipped (No players)

Processing Team: 1041
Skipped (No players)

Processing Team: 1048
Skipped (No players)

Processing Team: 105
Skipped (No players)

Processing Team: 1055
Skipped (No players)

Processing Team: 1062
Skipped (No players)

Processing Team: 1067
Skipped (No players)

Processing Team: 1069
Skipped (No players)

Processing Team: 1076
Skipped (No players)

Processing Team: 1081
Skipped (No players)

Processing Team: 1083
Skipped (No players)

Processing Team: 1090
Skipped (No players)

Processing Team: 1097
Skipped (No players

SystemExit: 0

C:\Users\Abishek\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import requests
import pandas as pd
import json
import time
import sys
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime

# ==============================
# CONFIGURATION
# ==============================

RAPID_API_KEY = "YOUR_API_KEY"

DB_USER = "root"
DB_PASSWORD = quote_plus("Vasu@76652")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "cricketdb"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True
)

session = requests.Session()

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# ==============================
# HELPER FUNCTIONS
# ==============================

def clean_dob(dob_string):
    if not dob_string:
        return None
    try:
        date_part = dob_string.split("(")[0].strip()
        parsed_date = datetime.strptime(date_part, "%B %d, %Y")
        return parsed_date.strftime("%Y-%m-%d")
    except:
        return None


def safe_api_call(url):
    try:
        response = session.get(url, headers=headers, timeout=20)

        if response.status_code == 429:
            return "RATE_LIMIT"

        if response.status_code == 204:
            return {}

        response.raise_for_status()
        return response.json()

    except:
        return None


# ==============================
# LOAD DATABASE STATE
# ==============================

# All Teams
df_teams = pd.read_sql("SELECT Team_ID FROM teams ORDER BY Team_ID", engine)
team_ids = df_teams["Team_ID"].astype(str).tolist()

# Existing Players
df_existing = pd.read_sql("SELECT Player_ID FROM players", engine)
existing_players = set(df_existing["Player_ID"].astype(str))

# Teams already processed (exist in mapping table)
df_team_done = pd.read_sql(
    "SELECT DISTINCT Team_ID FROM player_team",
    engine
)
completed_teams = set(df_team_done["Team_ID"].astype(str))

print(f"Total Teams: {len(team_ids)}")
print(f"Existing Players: {len(existing_players)}")
print(f"Completed Teams: {len(completed_teams)}")

# ==============================
# SQL STATEMENTS
# ==============================

insert_player_sql = """
INSERT INTO players
(Player_ID, Player_Name, Country, Role, Batting_Style, Bowling_Style, DOB, Meta)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    Player_Name=VALUES(Player_Name),
    Country=VALUES(Country),
    Role=VALUES(Role),
    Batting_Style=VALUES(Batting_Style),
    Bowling_Style=VALUES(Bowling_Style),
    DOB=VALUES(DOB),
    Meta=VALUES(Meta)
"""

insert_mapping_sql = """
INSERT IGNORE INTO player_team (Player_ID, Team_ID)
VALUES (%s, %s)
"""

# ==============================
# MAIN PROCESS
# ==============================

for TEAM_ID in team_ids:

    # 🚀 Skip already processed teams (NO API CALL)
    if TEAM_ID in completed_teams:
        print(f"Skipping Team {TEAM_ID} (Already Processed)")
        continue

    print(f"\nProcessing Team: {TEAM_ID}")

    team_url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{TEAM_ID}/players"
    team_data = safe_api_call(team_url)

    # 🚨 429 at team level
    if team_data == "RATE_LIMIT":
        print("🚨 429 Hit. Stopping safely.")
        sys.exit(0)

    # Skip empty / 204 / invalid
    if not team_data or "player" not in team_data or not team_data["player"]:
        print("Skipped (No players)")
        continue

    players_batch = []

    for p in team_data["player"]:

        player_id_raw = p.get("id")
        if not player_id_raw or not str(player_id_raw).isdigit():
            continue

        player_id = str(player_id_raw)

        # Skip if mapping already exists
        with engine.connect() as conn:
            result = conn.exec_driver_sql(
                "SELECT 1 FROM player_team WHERE Player_ID=%s AND Team_ID=%s LIMIT 1",
                (player_id, TEAM_ID)
            )
            if result.fetchone():
                continue

        # If player not already in players table → fetch detail
        if player_id not in existing_players:

            detail_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"
            detail = safe_api_call(detail_url)

            if detail == "RATE_LIMIT":
                print("🚨 429 Hit during player detail. Inserting collected data.")
                break

            if not detail:
                continue

            player_info = {
                "player_id": player_id,
                "player_name": detail.get("name") or p.get("name"),
                "country": detail.get("country") or detail.get("intlTeam"),
                "role": detail.get("role") or p.get("role"),
                "batting_style": detail.get("battingStyle") or p.get("battingStyle"),
                "bowling_style": detail.get("bowlingStyle") or p.get("bowlingStyle"),
                "dob": clean_dob(detail.get("DoB"))
            }

            players_batch.append(player_info)
            existing_players.add(player_id)

        else:
            # Only mapping required
            players_batch.append({
                "player_id": player_id,
                "existing_only": True
            })

        time.sleep(0.2)

    # ==========================
    # INSERT COLLECTED DATA
    # ==========================

    if players_batch:

        with engine.begin() as conn:
            for player in players_batch:

                pid = player["player_id"]

                if not player.get("existing_only"):
                    conn.exec_driver_sql(insert_player_sql, (
                        player["player_id"],
                        player.get("player_name"),
                        player.get("country"),
                        player.get("role"),
                        player.get("batting_style"),
                        player.get("bowling_style"),
                        player.get("dob"),
                        json.dumps(player)
                    ))

                conn.exec_driver_sql(insert_mapping_sql, (pid, TEAM_ID))

        print(f"Inserted {len(players_batch)} records for Team {TEAM_ID}")

        # Mark team as completed
        completed_teams.add(TEAM_ID)

print("\nAll teams processed successfully 🚀")


In [ ]:
import requests
import pandas as pd
import json
import time
import sys
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime

# ==============================
# CONFIGURATION
# ==============================

RAPID_API_KEY = "184ac0e286msh0297c81fa2ae2d1p17c3c1jsn2c791fe8e05d"

DB_USER = "root"
DB_PASSWORD = quote_plus("Vasu@76652")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "cricketdb"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True
)

session = requests.Session()

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# ==============================
# HELPER FUNCTIONS
# ==============================

def clean_dob(dob_string):
    if not dob_string:
        return None
    try:
        date_part = dob_string.split("(")[0].strip()
        parsed_date = datetime.strptime(date_part, "%B %d, %Y")
        return parsed_date.strftime("%Y-%m-%d")
    except:
        return None


def safe_api_call(url):
    try:
        response = session.get(url, headers=headers, timeout=20)

        if response.status_code == 429:
            return "RATE_LIMIT"

        if response.status_code == 204:
            return {}

        response.raise_for_status()
        return response.json()

    except:
        return None


# ==============================
# LOAD DATABASE STATE
# ==============================

df_teams = pd.read_sql("SELECT Team_ID FROM teams ORDER BY Team_ID", engine)
team_ids = df_teams["Team_ID"].astype(str).tolist()

df_existing_players = pd.read_sql("SELECT Player_ID FROM players", engine)
existing_players = set(df_existing_players["Player_ID"].astype(str))

df_completed_teams = pd.read_sql(
    "SELECT Team_ID FROM team_status WHERE Status IN ('COMPLETED','EMPTY')",
    engine
)
completed_teams = set(df_completed_teams["Team_ID"].astype(str))

print(f"Total Teams: {len(team_ids)}")
print(f"Existing Players: {len(existing_players)}")
print(f"Completed Teams: {len(completed_teams)}")

# ==============================
# SQL STATEMENTS
# ==============================

insert_player_sql = """
INSERT INTO players
(Player_ID, Player_Name, Country, Role, Batting_Style, Bowling_Style, DOB, Meta)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    Player_Name=VALUES(Player_Name),
    Country=VALUES(Country),
    Role=VALUES(Role),
    Batting_Style=VALUES(Batting_Style),
    Bowling_Style=VALUES(Bowling_Style),
    DOB=VALUES(DOB),
    Meta=VALUES(Meta)
"""

insert_mapping_sql = """
INSERT IGNORE INTO player_team (Player_ID, Team_ID)
VALUES (%s, %s)
"""

update_team_status_sql = """
INSERT INTO team_status (Team_ID, Status)
VALUES (%s, %s)
ON DUPLICATE KEY UPDATE Status=VALUES(Status)
"""

# ==============================
# MAIN LOOP
# ==============================

for TEAM_ID in team_ids:

    # Skip already completed teams (NO API CALL)
    if TEAM_ID in completed_teams:
        print(f"Skipping Team {TEAM_ID} (Already Done)")
        continue

    print(f"\nProcessing Team: {TEAM_ID}")

    team_url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{TEAM_ID}/players"
    team_data = safe_api_call(team_url)

    # 429 at team level
    if team_data == "RATE_LIMIT":
        print("🚨 429 Hit. Stopping safely.")
        sys.exit(0)

    # Skip empty teams
    if not team_data or "player" not in team_data or not team_data["player"]:
        print("No players. Marking team as EMPTY.")

        with engine.begin() as conn:
            conn.exec_driver_sql(update_team_status_sql, (TEAM_ID, "EMPTY"))

        continue

    players_batch = []

    for p in team_data["player"]:

        player_id_raw = p.get("id")

        if not player_id_raw or not str(player_id_raw).isdigit():
            continue

        player_id = str(player_id_raw)

        # Skip if mapping already exists
        with engine.connect() as conn:
            result = conn.exec_driver_sql(
                "SELECT 1 FROM player_team WHERE Player_ID=%s AND Team_ID=%s LIMIT 1",
                (player_id, TEAM_ID)
            )
            if result.fetchone():
                continue

        # Fetch player detail only if not exists
        if player_id not in existing_players:

            detail_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"
            detail = safe_api_call(detail_url)

            if detail == "RATE_LIMIT":
                print("🚨 429 Hit during player detail.")
                break

            if not detail:
                continue

            player_info = {
                "player_id": player_id,
                "player_name": detail.get("name") or p.get("name"),
                "country": detail.get("country") or detail.get("intlTeam"),
                "role": detail.get("role") or p.get("role"),
                "batting_style": detail.get("battingStyle") or p.get("battingStyle"),
                "bowling_style": detail.get("bowlingStyle") or p.get("bowlingStyle"),
                "dob": clean_dob(detail.get("DoB"))
            }

            players_batch.append(player_info)
            existing_players.add(player_id)

        else:
            players_batch.append({
                "player_id": player_id,
                "existing_only": True
            })

        time.sleep(0.2)

    # ==========================
    # INSERT COLLECTED DATA
    # ==========================

    if players_batch:

        with engine.begin() as conn:
            for player in players_batch:

                pid = player["player_id"]

                if not player.get("existing_only"):
                    conn.exec_driver_sql(insert_player_sql, (
                        player["player_id"],
                        player.get("player_name"),
                        player.get("country"),
                        player.get("role"),
                        player.get("batting_style"),
                        player.get("bowling_style"),
                        player.get("dob"),
                        json.dumps(player)
                    ))

                conn.exec_driver_sql(insert_mapping_sql, (pid, TEAM_ID))

        print(f"Inserted {len(players_batch)} players for Team {TEAM_ID}")

        # Mark team completed
        with engine.begin() as conn:
            conn.exec_driver_sql(update_team_status_sql, (TEAM_ID, "COMPLETED"))

    # If 429 during player fetch
    if team_data == "RATE_LIMIT":
        break


print("\nAll teams processed successfully 🚀")


Total Teams: 924
Existing Players: 262
Completed Teams: 0

Processing Team: 10

Processing Team: 100

Processing Team: 101
No players. Marking team as EMPTY.

Processing Team: 1011
No players. Marking team as EMPTY.

Processing Team: 1018
No players. Marking team as EMPTY.

Processing Team: 1020
No players. Marking team as EMPTY.

Processing Team: 1027
No players. Marking team as EMPTY.

Processing Team: 103
No players. Marking team as EMPTY.

Processing Team: 1032
No players. Marking team as EMPTY.

Processing Team: 1034
No players. Marking team as EMPTY.

Processing Team: 104
No players. Marking team as EMPTY.

Processing Team: 1041
No players. Marking team as EMPTY.

Processing Team: 1048
No players. Marking team as EMPTY.

Processing Team: 105
No players. Marking team as EMPTY.

Processing Team: 1055
No players. Marking team as EMPTY.

Processing Team: 1062
No players. Marking team as EMPTY.

Processing Team: 1067
No players. Marking team as EMPTY.

Processing Team: 1069
No players

SystemExit: 0

C:\Users\Abishek\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import requests
import pandas as pd
import json
import time
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime

# ==============================
# CONFIGURATION
# ==============================

RAPID_API_KEY = "bfed77adbdmshfe2205d44843f14p175691jsnc2b870a80115"

DB_USER = "root"
DB_PASSWORD = quote_plus("Vasu@76652")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "cricketdb"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True
)

session = requests.Session()

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# ==============================
# HELPER
# ==============================

def clean_dob(dob_string):
    if not dob_string:
        return None
    try:
        date_part = dob_string.split("(")[0].strip()
        parsed_date = datetime.strptime(date_part, "%B %d, %Y")
        return parsed_date.strftime("%Y-%m-%d")
    except:
        return None


def safe_api_call(url):
    try:
        response = session.get(url, headers=headers, timeout=20)

        if response.status_code == 429:
            return "RATE_LIMIT"

        if response.status_code == 204:
            return {}

        response.raise_for_status()
        return response.json()

    except:
        return None


# ==============================
# LOAD DB STATE
# ==============================

team_ids = pd.read_sql(
    "SELECT Team_ID FROM teams ORDER BY Team_ID",
    engine
)["Team_ID"].astype(str).tolist()

existing_players = set(
    pd.read_sql("SELECT Player_ID FROM players", engine)
    ["Player_ID"].astype(str)
)

completed_teams = set(
    pd.read_sql(
        "SELECT Team_ID FROM team_status WHERE Status IN ('COMPLETED','EMPTY')",
        engine
    )["Team_ID"].astype(str)
)

print(f"\nTotal Teams: {len(team_ids)}")
print(f"Existing Players: {len(existing_players)}")
print(f"Completed Teams: {len(completed_teams)}")

# ==============================
# SQL
# ==============================

insert_player_sql = """
INSERT INTO players
(Player_ID, Player_Name, Country, Role, Batting_Style, Bowling_Style, DOB, Meta)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    Player_Name=VALUES(Player_Name),
    Country=VALUES(Country),
    Role=VALUES(Role),
    Batting_Style=VALUES(Batting_Style),
    Bowling_Style=VALUES(Bowling_Style),
    DOB=VALUES(DOB),
    Meta=VALUES(Meta)
"""

insert_mapping_sql = """
INSERT IGNORE INTO player_team (Player_ID, Team_ID)
VALUES (%s, %s)
"""

update_team_status_sql = """
INSERT INTO team_status (Team_ID, Status)
VALUES (%s, %s)
ON DUPLICATE KEY UPDATE Status=VALUES(Status)
"""

# ==============================
# MAIN LOOP
# ==============================

for TEAM_ID in team_ids:

    if TEAM_ID in completed_teams:
        print(f"⏭️ Skipping Team {TEAM_ID} (Already Completed)")
        continue

    print(f"\n🔵 Processing Team: {TEAM_ID}")

    team_url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{TEAM_ID}/players"
    team_data = safe_api_call(team_url)

    if team_data == "RATE_LIMIT":
        print("🚨 429 Rate Limit Hit. Stopping.")
        break

    if not team_data or "player" not in team_data or not team_data["player"]:
        print(f"⚪ Team {TEAM_ID} has NO players → Marking EMPTY")

        with engine.begin() as conn:
            conn.exec_driver_sql(update_team_status_sql, (TEAM_ID, "EMPTY"))

        continue

    for p in team_data["player"]:

        player_id_raw = p.get("id")

        if not player_id_raw or not str(player_id_raw).isdigit():
            continue

        player_id = str(player_id_raw)

        # Check mapping
        with engine.connect() as conn:
            result = conn.exec_driver_sql(
                "SELECT 1 FROM player_team WHERE Player_ID=%s AND Team_ID=%s LIMIT 1",
                (player_id, TEAM_ID)
            )
            if result.fetchone():
                print(f"   ⏭️ Skipping Mapping → Player {player_id} already mapped to Team {TEAM_ID}")
                continue

        if player_id in existing_players:
            print(f"   ⏭️ Skipping Player {player_id} (Already Exists)")
        else:
            print(f"   ➕ Fetching & Inserting Player {player_id}")

            detail_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{player_id}"
            detail = safe_api_call(detail_url)

            if detail == "RATE_LIMIT":
                print("🚨 429 Hit during player detail. Stopping safely.")
                break

            if not detail:
                print(f"   ⚠️ Detail fetch failed for {player_id}")
                continue

            player_info = {
                "player_id": player_id,
                "player_name": detail.get("name") or p.get("name"),
                "country": detail.get("country") or detail.get("intlTeam"),
                "role": detail.get("role") or p.get("role"),
                "batting_style": detail.get("battingStyle") or p.get("battingStyle"),
                "bowling_style": detail.get("bowlingStyle") or p.get("bowlingStyle"),
                "dob": clean_dob(detail.get("DoB"))
            }

            with engine.begin() as conn:
                conn.exec_driver_sql(insert_player_sql, (
                    player_info["player_id"],
                    player_info["player_name"],
                    player_info["country"],
                    player_info["role"],
                    player_info["batting_style"],
                    player_info["bowling_style"],
                    player_info["dob"],
                    json.dumps(player_info)
                ))

            existing_players.add(player_id)

        # Insert mapping always
        with engine.begin() as conn:
            conn.exec_driver_sql(insert_mapping_sql, (player_id, TEAM_ID))

        print(f"   ✅ Mapping inserted → Player {player_id} → Team {TEAM_ID}")

        time.sleep(0.2)

    # Mark team completed
    with engine.begin() as conn:
        conn.exec_driver_sql(update_team_status_sql, (TEAM_ID, "COMPLETED"))

    print(f"✅ Team {TEAM_ID} Completed")

print("\n🏁 Process Finished")



Total Teams: 924
Existing Players: 1560
Completed Teams: 924
⏭️ Skipping Team 10 (Already Completed)
⏭️ Skipping Team 100 (Already Completed)
⏭️ Skipping Team 101 (Already Completed)
⏭️ Skipping Team 1011 (Already Completed)
⏭️ Skipping Team 1018 (Already Completed)
⏭️ Skipping Team 1020 (Already Completed)
⏭️ Skipping Team 1027 (Already Completed)
⏭️ Skipping Team 103 (Already Completed)
⏭️ Skipping Team 1032 (Already Completed)
⏭️ Skipping Team 1034 (Already Completed)
⏭️ Skipping Team 104 (Already Completed)
⏭️ Skipping Team 1041 (Already Completed)
⏭️ Skipping Team 1048 (Already Completed)
⏭️ Skipping Team 105 (Already Completed)
⏭️ Skipping Team 1055 (Already Completed)
⏭️ Skipping Team 1062 (Already Completed)
⏭️ Skipping Team 1067 (Already Completed)
⏭️ Skipping Team 1069 (Already Completed)
⏭️ Skipping Team 1076 (Already Completed)
⏭️ Skipping Team 1081 (Already Completed)
⏭️ Skipping Team 1083 (Already Completed)
⏭️ Skipping Team 1090 (Already Completed)
⏭️ Skipping Team 1097